# 08: The stock global fits — the machinery

**This notebook targets the development stack (LATW `dev` branch).** It
uses the *installed* stock global fit from the development LISA Analysis
Tools packages (`LISAanalysistools/install.sh`), not the pip releases.
No Colab:

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
os.environ.setdefault("MAKE_PLOTS", "0")   # we do not need in-run eryn plots here

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
import dataclasses
from lisatools.utils.constants import *

from lisatools.globalfit.stock import erebor

cupy not found, using numpy instead. This will be very slow for large runs. Please install cupy and a compatible CUDA version for GPU acceleration.


[`01`](01_GlobalFitQuickstart.ipynb) ran a stock global fit in four
lines and read its products back. This notebook opens the **machinery**
underneath that one-liner: the variant catalogue, the data-in
preprocessing layer, the layered settings the run is built from, the
declarative *recipe* of sampler moves, and — the part you will reach for
when your research needs something new — **how to write your own move
module and drop it into the stack.**

We demo on the lightest variant, **`gb_no_fg_lite`** (a GB-only fit).
[`02`](02_StockGlobalFitGallery.ipynb) is the per-variant gallery;
[`00`](00_SetupAndAtlas.ipynb) places `lisatools.globalfit` in the stack.

### How to read this notebook

Each `##` section opens with a **TL;DR** and one minimal cell; drop into
*"Going deeper"* for the internals. Most cells here are *inspection* (a
stock fit is a cheap Python object until you `build()` it), so they run
instantly — only §5 actually builds and runs a short chain.

## 1. The variant catalogue

**TL;DR.** A *stock global fit* is an installed, versioned run recipe — a
`StockGlobalFit` subclass, not a settings file. The **erebor** family
registers several; `erebor.get_stock_options()` lists them, and each is
reachable as a module attribute (`erebor.gb_no_fg`) or via
`erebor.get_stock("gb_no_fg")`.

In [2]:
for name, desc in erebor.get_stock_options():
    print(f"{name:24s} {desc[:64]}")

gb_no_fg                 GB-only fit on the mojito L1 GB galaxy: fixed PSD, no foreground
all_sources              All six branches (gb, psd, galfor, mbh, emri, sobbh) composed fr
full_year_combined       Catalogue-driven multi-leaf MBH+EMRI+SOBBH fit over the full yea
noise_only               Joint WDM noise fit: instrument PSD + hyperbolic-tangent galacti
noise_sgwb               Joint WDM noise fit: instrument PSD + galactic foreground + powe
gb_no_fg_lite            Laptop-smoke twin of gb_no_fg: two-week span, 10 iterations, 4 w
all_sources_lite         Laptop-smoke twin of all_sources: the same six branches and mach
full_year_combined_lite  Laptop-smoke twin of full_year_combined: one month instead of a 
noise_only_lite          Laptop-smoke twin of noise_only: quarter-length time grid, 10 it
noise_sgwb_lite          Laptop-smoke twin of noise_sgwb: quarter-length time grid, 10 it


Five base variants, each with a `*_lite` laptop-smoke twin:

| variant | what it fits | when to use it |
|---|---|---|
| **`gb_no_fg`** | galactic binaries only, fixed PSD, f > 6 mHz | GB methods; the simplest full run |
| **`all_sources`** | all six branches (gb, psd, galfor, mbh, emri, sobbh) | the real joint global fit |
| **`full_year_combined`** | multi-leaf MBH + EMRI + SOBBH, full year | the heavy-source branches |
| **`noise_only`** | instrument PSD + galactic foreground | noise-model work |
| **`noise_sgwb`** | PSD + foreground + power-law SGWB | stochastic-background work |

`erebor.get_stock(name, **overrides)` and `erebor.<name>(**overrides)` are
equivalent constructors. The per-variant deep demos are in
[`02`](02_StockGlobalFitGallery.ipynb).

In [3]:
fit = erebor.gb_no_fg_lite()      # our demo subject for the rest of the notebook
print(fit.describe())

GBNoForegroundLiteGlobalFit (gb_no_fg_lite) — Laptop-smoke twin of gb_no_fg: two-week span, 10 iterations, 4 walkers x 2 temps, 2 GB repeat proposals, CPU.
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 2.5
    num_iterations = 10
    file_store_dir = './gf_output_gb_no_fg/'
    base_file_name = 'gb_no_fg_test_2'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 1209600.0
    min_freq = 0.006
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.05
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['gb']
    gb: GBNoFgGBSettings
  recipe:
    [pe] gb_pe:
        rj_prior  <- stock branch=gb
  setup_function: setup_recipe


## 2. Data in — the preprocessing layer

**TL;DR.** Every variant turns raw input into the analysis data through a
*data processor*. `lisatools.globalfit.preprocessing` holds the loaders
(`L1DataLoader` for mojito L1, `SangriaDataLoader` for the LDC file) and
the in-process `SyntheticDataProcessor`; `stock/erebor/injections.py` adds
the synthetic per-source stream builders. One knob — `general.data_mode`
(env `DATA_PROCESSOR`) — swaps the whole pipeline.

In [4]:
from lisatools.globalfit import preprocessing
loaders = [c for c in dir(preprocessing)
           if c.endswith(("DataLoader", "ProcessingStep", "DataProcessor"))]
print("preprocessing:", loaders)

# the data-mode knob and its default for this variant
print("\ngb_no_fg default data_mode:", fit.general.data_mode)   # "mojito"

preprocessing: ['BaseProcessingStep', 'L1DataLoader', 'L1ProcessingStep', 'SangriaDataLoader', 'SangriaProcessingStep', 'SyntheticDataProcessor', 'SyntheticSourceProcessingStep']

gb_no_fg default data_mode: mojito


### Going deeper: the three data modes and the synthetic fallback

- **`"mojito"`** (default for `gb_no_fg` / `all_sources` /
  `full_year_combined`) — `L1DataLoader` reads a mojito L1 folder
  (per-link time series + source catalogues), builds TDI, and pours it
  onto the analysis grid. For GB it also populates
  `general_info.catalogue["GB"]` so leaves can start at true catalogue
  points (the SNR-cut subset).
- **`"synthetic"`** — no external data: the processor builds each present
  branch's stream *in-process* from injection tables. The tables come
  from `stock/erebor/injections.py` (`make_gb_injections`,
  `make_emri_injections`, `make_mbh_injections`, `make_sobbh_injections`)
  in either `"stock"` (fixed) or `"prior"` (seeded prior draws) mode.
- **`"sangria"`** — `SangriaDataLoader` on the legacy LDC training file
  (`all_sources` only).

**The fallback:** when `data_mode="mojito"` but the mojito folder is
missing, `build()` switches to `"synthetic"` with prior-drawn injections
automatically (`EreborFit.resolve_data_source`), so a bare laptop still
runs. Swap the mode by one assignment:

```python
fit.general.data_mode = "synthetic"   # or DATA_PROCESSOR=synthetic
```

An explicit `general.data_processor_class` swap always wins over the
`data_mode` knob — that is the seam for plugging in your own loader.

## 3. The settings pyramid + env resolution

**TL;DR.** A fit is a three-level pyramid: the **general block**
(`fit.general` — grid, band, data mode, run shape) sits over the
**per-branch blocks** (`fit.gb`, …) which sit over the **recipe**
(`fit.recipe` — the move stack). Substituting a level replaces everything
beneath it and nothing above. Many field defaults are
**environment-backed**, resolving *explicit kwarg > env var > hard
default*.

In [5]:
# the pyramid, top to bottom
print("general :", type(fit.general).__name__)
print("branches:", {n: type(getattr(fit, n)).__name__ for n in fit.branches})
print("recipe  :", fit.recipe)

# env resolution: explicit kwarg > env var > hard default
os.environ["NWALKERS"] = "6"
print("\nenv NWALKERS=6      -> nwalkers =", erebor.gb_no_fg().general.nwalkers)
print("kwarg nwalkers=10   -> nwalkers =", erebor.gb_no_fg(nwalkers=10).general.nwalkers)
del os.environ["NWALKERS"]
print("unset (hard default)-> nwalkers =", erebor.gb_no_fg().general.nwalkers)

general : GBNoFgGeneralSettings
branches: {'gb': 'GBNoFgGBSettings'}
recipe  : RecipeSpec(['gb_pe'])

env NWALKERS=6      -> nwalkers = 4
kwarg nwalkers=10   -> nwalkers = 10
unset (hard default)-> nwalkers = 4


### Going deeper: `env_default` / `env_resolve`

The env-backed dataclass fields use
`lisatools.globalfit.stock.base.env_default(VAR, default, cast)` as their
`default_factory`: the factory reads the environment **at construction
time**, so `GF_NUM_ITER`, `NWALKERS`, `NTEMPS`, `TOBS_TARGET`,
`DATA_PROCESSOR`, `USE_GPU`/`GPUS`/`GPU_BACKEND`, `FILE_STORE_DIR` all just
seed the corresponding knob's default. Because they are *only* defaults,
an explicit constructor kwarg or a later `fit.general.x = ...` assignment
always wins. (`*_lite` twins go one step further and pin a fixed
smoke-size preset on top of the env defaults — see
[`01` §Lite ↔ heavy](01_GlobalFitQuickstart.ipynb).)

## 4. Recipes — stages and moves

**TL;DR.** `fit.recipe` is a `RecipeSpec`: an ordered list of `StageSpec`
stages, each holding `MoveSpec` move descriptions. They are *declarative
specifications* (picklable, cheap) — the real sampler moves are
constructed only at build time, when the variant's `setup_function` and
`materialize_recipe` turn each `MoveSpec` into a live move. Edit the stack
with `add_move` / `pop_move` / `add_stage` / `pop_stage` /
`set_move_debug`, all before `build()`.

In [6]:
from lisatools.globalfit.stock.base import MoveSpec, StageSpec

demo = erebor.gb_no_fg_lite()
print("before:")
print(demo.list_moves())

# add a stock move to the gb_pe stage (an f-statistic MCMC refinement move),
# then take it back out — pure spec editing, no build
demo.recipe.add_move(MoveSpec("rj_fstat_mcmc", branch="gb"), stage="gb_pe")
print("\nafter add_move:")
print(demo.list_moves())

demo.recipe.pop_move("rj_fstat_mcmc")
print("\nafter pop_move:", demo.recipe.move_names())

before:
[pe] gb_pe:
    rj_prior  <- stock branch=gb

after add_move:
[pe] gb_pe:
    rj_prior  <- stock branch=gb
    rj_fstat_mcmc  <- stock branch=gb

after pop_move: ['rj_prior']


### Going deeper: `MoveSpec`, `StageSpec`, `materialize_recipe`

- **`MoveSpec(name, target=None, kwargs={}, branch=None, instance=None,
  debug=None)`** — one move. `target=None` means "the variant's
  `setup_function` supplies the move under this `name`" (the stock path,
  e.g. `"rj_prior"`, `"psd_pe"`, `"mbh_pe"`). A **callable** `target` is
  invoked `target(ctx, **kwargs)` at build (this is the plug-in seam,
  §5); **`instance=`** takes a fully-built move (but may break pickling).
- **`StageSpec(name, kind, moves, step_kwargs, combine_kwargs)`** — one
  recipe stage; `kind` is `"search"` / `"pe"` / `"rj"`, selecting the
  RecipeStep class.
- **`materialize_recipe`** (in `stock/base.py`) walks the spec at build
  time: per move it resolves `instance` > `target` > the stock builder,
  wraps each stage's moves in a `GFCombineMove`, and adds them to the
  runtime `Recipe`. The stock move builders (`build_gb_moves`,
  `build_psd_moves`, …) construct exactly the named moves.

Full reference:
`LISAanalysistools/docs/stock-stages-and-moves.md`.

## 5. Write your own module

**TL;DR.** The move is the plug-in point of the whole global fit. A
global-fit module is **any object with `propose(model, state) ->
(new_state, accepted)`**. It receives the `model` (whose
`analysis_container_arr` holds the per-walker residuals) and the sampler
`State`; it may read or *edit the residuals / state in place* so the next
module sees the update; it returns the (possibly changed) state and an
`accepted` boolean array. **It need not be an MCMC proposal** — it is just
inputs in, outputs out, and you can do literally anything in between.

That flexibility is the point worth dwelling on. A `propose` can be:

- a **diagnostics dumper** — read the residual, record or plot a statistic,
  return the state untouched (what we build below);
- a **residual surgeon** — subtract a known/external template from every
  walker's residual so downstream moves fit what is left;
- a **deterministic annealer / rescaler** — nudge coordinates on a fixed
  schedule with no accept/reject;
- a **bridge to an external code** — hand the current residual to another
  sampler or an ML model and fold its answer back into the state.

None of these are Metropolis proposals; all are legal modules because the
contract is only `propose(model, state) -> (state, accepted)`. Here is a
minimal, non-MCMC one — it records `sum |r|^2` of walker 0's residual each
time it runs and returns the state unchanged:

In [7]:
from eryn.moves import Move
from lisatools.globalfit.moves.globalfitmove import GlobalFitMove

class ResidualStatLogger(GlobalFitMove, Move):
    """A no-op global-fit module: record a residual statistic each step.

    Demonstrates the plug-in contract. It is NOT an MCMC proposal: it reads
    the current per-walker residual off the model's AnalysisContainerArray,
    appends a scalar summary to ``self.history``, and returns the state
    unchanged with an all-False ``accepted`` (nothing was proposed).
    """

    def __init__(self, name="residual_stat_logger", **kwargs):
        Move.__init__(self, **kwargs)          # eryn Move machinery
        GlobalFitMove.__init__(self, name=name)  # global-fit name/bookkeeping
        self.history = []

    def propose(self, model, state):
        acs = model.analysis_container_arr       # per-walker residual slabs
        resid0 = np.asarray(acs[0].data.arr)     # walker 0's residual (XYZ, WDM)
        stat = float(np.sum(np.abs(resid0) ** 2))  # ~ -2 * source-only logL
        self.history.append(stat)
        self.num_proposals += 1
        print(f"[ResidualStatLogger] step {len(self.history)}: "
              f"walker-0 sum|r|^2 = {stat:.6e}")
        ntemps, nwalkers = state.log_like.shape[:2]
        accepted = np.zeros((ntemps, nwalkers), dtype=bool)   # nothing proposed
        return state, accepted


def build_residual_stat_logger(ctx, **kwargs):
    """MoveSpec factory. ``ctx`` is the MoveBuildContext (recipe, engine_info,
    curr, acs, priors, state) handed to every target at materialize time; a
    real module would read what it needs from it. We ignore it here."""
    return ResidualStatLogger(**kwargs)

print("defined ResidualStatLogger + factory")

defined ResidualStatLogger + factory


Now wire it into a `gb_no_fg_lite` recipe with a `MoveSpec` whose
`target` is our factory, and run a short chain. We shrink the fit hard
(synthetic in-process data, a small RJ leaf cap, one repeat proposal, two
iterations) purely so the demo runs in a couple of minutes — the module
itself is grid-agnostic.

In [8]:
fit5 = erebor.gb_no_fg_lite()
fit5.general.data_mode = "synthetic"   # in-process GB stream, no external data
fit5.general.num_iterations = 2        # explicit: overrides the lite preset's 10
fit5.gb.nleaves_max = 4                 # cap the RJ leaf budget for speed
fit5.gb.num_repeat_proposals = 1        # one in-model repeat per iteration

# drop our module into the gb_pe stage, right after the stock GB move
fit5.recipe.add_move(
    MoveSpec("residual_stat_logger", target=build_residual_stat_logger),
    stage="gb_pe",
)
print(fit5.list_moves())

[pe] gb_pe:
    rj_prior  <- stock branch=gb
    residual_stat_logger  <- build_residual_stat_logger


In [9]:
curr5 = fit5.build()
fit5.run()

2026-07-12 22:27:49,183 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_gb_no_fg/gb_no_fg_test_2_testing.h5


2026-07-12 22:27:49,185 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_gb_no_fg/gb_no_fg_test_2_artifacts/


2026-07-12 22:27:56,434 - GeneralSetup - INFO - Using fixed PSD kwargs: {'psd_params': [1.5e-11, 3e-15], 'galfor_params': None}


2026-07-12 22:27:56,446 - GeneralSetup - DEBUG - Preprocess setting: plot_folder = ./gf_output_gb_no_fg/gb_no_fg_test_2_artifacts/


2026-07-12 22:27:56,447 - GeneralSetup - DEBUG - Preprocess setting: highpass_kwargs = None


...

1it [00:06,  6.78s/it]

2it [00:06,  3.39s/it]


100%|██████████| 2/2 [00:16<00:00,  7.86s/it]

100%|██████████| 2/2 [00:16<00:00,  8.09s/it]

[ResidualStatLogger] step 2: walker-0 sum|r|^2 = 4.889029e-40
2026-07-12 22:28:28,326 - lisatools.globalfit.run - INFO - Residuals saved.


The `[ResidualStatLogger]` lines interleaved in the run output above are
our module firing once per sampler iteration — proof it is wired into the
live stack. The `self.history` list accumulates on the materialized move
object inside the run; in your own driver you would hold that reference
(or read the module's move off the sampler) to pull the recorded values
back out afterward. The `MoveSpec` still sits in the recipe:

In [10]:
print("our module in the recipe:", fit5.recipe.get_move("residual_stat_logger"))
print("full stack:\n" + fit5.list_moves())

our module in the recipe: MoveSpec(name='residual_stat_logger', target=<function build_residual_stat_logger at 0x133e472e0>, kwargs={}, branch=None, weight=None, instance=None, debug=None)
full stack:
[pe] gb_pe:
    rj_prior  <- stock branch=gb
    residual_stat_logger  <- build_residual_stat_logger


### Going deeper: what the contract buys you

- **`model.analysis_container_arr`** is an `AnalysisContainerArray` — one
  `AnalysisContainer` per walker, each holding that walker's residual
  (`acs[w].data.arr`) and its sensitivity. Editing `acs[w]` in place is
  how a "residual surgeon" module changes what later modules fit.
- The returned **`state`** is an eryn `State`; return it unchanged for a
  read-only module, or `deepcopy` and edit `state.branches[...]` for a
  module that moves coordinates. **`accepted`** is `(ntemps, nwalkers)`
  and feeds the acceptance bookkeeping — all-`False` for a module that
  proposes nothing.
- Because it is *just* `propose(model, state)`, the same object also
  satisfies eryn's move interface, so it stacks in a `StageSpec`
  alongside the stock moves and is combined by `GFCombineMove` in order.
  The stock `ResidualAddOneRemoveOneMove`
  (`globalfit/moves/addremovemove.py`) is a full-featured example of the
  same contract — remove each source from the residual, resample it, add
  it back.

## 6. Instrumentation

**TL;DR.** Debug-capable moves carry a uniform `set_debug` hook; the
recipe exposes it as `fit.set_move_debug(name, ...)` and
`fit.set_stage_debug(stage, ...)`, which flip on a per-move
**residual-trace flip-book** at materialize time. The run persists to the
eryn HDF5 backend (§`01`) and the `postprocessing` /
`diagnosticplot` modules read it back.

In [11]:
demo6 = erebor.gb_no_fg_lite()

# arm one move's debug instrumentation (writes per-leaf residual-trace PNGs)
demo6.set_move_debug("rj_prior", plot_dir="./gb_debug", every=5)
print("rj_prior debug spec:", demo6.recipe.get_move("rj_prior").debug)

# ...or a whole stage at once (each move that has no per-move override)
demo6.set_stage_debug("gb_pe", plot_walker=0)
print("gb_pe stage debug spec:", demo6.recipe._stage("gb_pe").debug)

rj_prior debug spec: {'enabled': True, 'plot_dir': './gb_debug', 'every': 5}
gb_pe stage debug spec: {'enabled': True, 'plot_walker': 0}


### Going deeper: the flip-book and the readout path

- **`set_move_debug` / `set_stage_debug`** stash a `debug` payload on the
  `MoveSpec` / `StageSpec`; `materialize_recipe` applies it via the move's
  `GlobalFitMove.set_debug(enabled, plot_dir=, plot_walker=, plot_leaf=,
  plot_band=, every=)`. The GB special-stretch move and
  `ResidualAddOneRemoveOneMove` then emit a per-step
  `[template | data | residual]` flip-book so you can watch a source
  leave and re-enter the residual. Precedence: per-move spec > stage spec
  > the move's `{BRANCH}_DEBUG` env default. Moves without a debug hook
  (e.g. `PSDMove`) carry the flag inertly.
- **Readout:** the run writes a `GFHDFBackend` HDF5 file
  (`curr.backend`, reopenable with `GFHDFBackend(path)`) plus the
  `…_artifacts/` folder (run log, `dump_settings` summary, diagnostic
  plots). `lisatools.globalfit.postprocessing` builds catalogues from the
  chains; `lisatools.globalfit.diagnosticplot` draws the trace/corner/
  log-likelihood figures (see [`01` §Diagnostic plots](01_GlobalFitQuickstart.ipynb)).

## Where to go next

- [`02`](02_StockGlobalFitGallery.ipynb) — a deep, run-it demo of every
  stock variant, branch by branch.
- [`06`](06_ErynSmallToLarge.ipynb) — the eryn sampler the recipe drives
  (walkers, tempering, reversible jump).
- `LISAanalysistools/docs/global-fit-launch.md` (running on ranks/GPUs)
  and `docs/stock-stages-and-moves.md` (the recipe layer).